# Diabetes — Preprocessing
Cleans the raw data (fixes hidden zeros, imputes, scales)
and saves a ready-to-train version plus train/test splits.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


## 1. Load raw data

In [2]:
df = pd.read_csv("../../data/raw/diabetes.csv")
df.shape


(768, 9)

## 2. Fix hidden missing values
Glucose, BloodPressure, SkinThickness, Insulin, and BMI can't
medically be 0 — those zeros are actually missing data.
We replace them with NaN, then impute using the median.

In [3]:
cols_with_hidden_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[cols_with_hidden_zeros] = df[cols_with_hidden_zeros].replace(0, np.nan)

print("Missing values per column:")
print(df.isnull().sum())


Missing values per column:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [4]:
for col in cols_with_hidden_zeros:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Confirm no missing values remain
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


## 3. Separate features (X) and target (y)

In [5]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33


## 4. Train/test split
80% for training, 20% held out for testing.
`stratify=y` keeps the same diabetic/non-diabetic ratio in both sets.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (614, 8)
Test shape: (154, 8)


## 5. Scale the features
Fit the scaler on the TRAINING data only, then apply it to both
sets. This avoids leaking test-set information into training.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

X_train_scaled.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.851355,-1.056427,-0.826740,-1.918187,-1.203361,-0.769477,0.310794,-0.792169
711,0.356576,0.144399,0.477772,-0.229874,-1.470195,-0.417498,-0.116439,0.561034
373,-0.549372,-0.556083,-1.152868,1.233330,-0.555335,0.359790,-0.764862,-0.707594
46,-0.851355,0.811525,-1.315932,-0.004766,-0.161437,-0.402832,0.262314,-0.369293
682,-1.153338,-0.889646,-0.663676,1.120776,-0.415565,1.782373,-0.337630,-0.961320


## 6. Save everything for the next notebook (model training)
- Cleaned full dataset -> data/processed/diabetes_cleaned.csv
- Scaler object -> models/diabetes_scaler.pkl (needed later to scale
  new patient input the same way, at prediction time)

In [8]:
# Save the cleaned (pre-split) dataset
df.to_csv("../../data/processed/diabetes_cleaned.csv", index=False)

# Save the scaler for reuse in the Flask app
joblib.dump(scaler, "../../models/diabetes_scaler.pkl")

# Save the train/test splits so 03_model_training.ipynb can load them directly
X_train_scaled.to_csv("../../data/processed/diabetes_X_train.csv", index=False)
X_test_scaled.to_csv("../../data/processed/diabetes_X_test.csv", index=False)
y_train.to_csv("../../data/processed/diabetes_y_train.csv", index=False)
y_test.to_csv("../../data/processed/diabetes_y_test.csv", index=False)

print("Saved cleaned data, scaler, and train/test splits.")


Saved cleaned data, scaler, and train/test splits.


## Next step
Open **03_model_training.ipynb** to train and compare models
using these saved train/test files.